# CLERC retrieval baselines (DPR) — Kaggle kernel

Runs the same pipeline as `setup/run_baselines.py` from **Domain-Specific RL Retrieval with Limited Labels**.

## Before you run

1. **Settings → Internet** — ON (needed for Hugging Face: CLERC data + `LegalBERT-DPR-CLERC-ft`).
2. **Accelerator** — optional **GPU** speeds up DPR encoding (CPU works, slower).
3. **Repo source** — either:
   - **Clone** (cell below): set `REPO_URL` to your public GitHub repo, **or**
   - **Add Dataset**: zip this project, upload as a Kaggle Dataset, then copy it into `/kaggle/working` and skip the clone cell.
4. Optional: add a **Kaggle Secret** `HF_TOKEN` for higher Hugging Face rate limits.

## Evaluation note

DPR retrieves **passage IDs** → use **`--qrels-level passage`** (enforced by the runner when not using `--mock`).

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# --- edit if needed ---
REPO_URL = "https://github.com/ali-salloum6/Domain-Specific-RL-Retrieval-with-Limited-Labels.git"
REPO_DIR_NAME = "Domain-Specific-RL-Retrieval-with-Limited-Labels"

# Hugging Face + project caches (Kaggle: /kaggle/working; local: ./kaggle_work)
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path.cwd() / "kaggle_work")
WORK.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK / "hf_home")
os.environ["HF_DATASETS_CACHE"] = str(WORK / "hf_datasets")
os.environ["CLERC_HF_CACHE"] = str(WORK / "clerc_hf_cache")
os.environ["TRANSFORMERS_CACHE"] = str(WORK / "hf_home" / "hub")

# Optional: HF token from Kaggle Secrets (Settings → Secrets → name: HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except ImportError:
    print("Not on Kaggle kernel (kaggle_secrets unavailable); set HF_TOKEN in env if needed")
except Exception as e:
    print("No HF_TOKEN secret (optional):", e)

for k in ("HF_HOME", "HF_DATASETS_CACHE", "CLERC_HF_CACHE"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)
print("WORK:", WORK)

## 2. Get project code

Skip this section if you added the repo as a **Kaggle Dataset** and copied it to `/kaggle/working`.

In [ ]:
import os
import subprocess
from pathlib import Path

dest = WORK / REPO_DIR_NAME
if dest.exists():
    print("Repo folder already exists:", dest)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)

os.chdir(dest)
print("CWD:", Path.cwd())

## 3. System packages (Java — only if you plan Pyserini / BM25)

DPR-only runs **do not** need Java. Uncomment if you will build or use a Lucene index.

In [ ]:
# !apt-get update -qq && apt-get install -y -qq openjdk-11-jdk-headless
# import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
print("Skip Java unless using BM25/Pyserini indexing.")

## 4. Python dependencies

In [ ]:
!pip install -q -r setup/requirements.txt

## 5. Sanity check imports

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from setup.clerc_data import load_queries_and_qrels_hf
from setup.run_baselines import run_dpr
from setup.metrics import evaluate

print("ROOT:", ROOT)
print("OK: setup package imports")

## 6. Run DPR baseline (passage-level qrels)

Adjust **`DPR_CORPUS`** and **`MAX_QUERIES`** for speed vs. quality. First run downloads CLERC shards + the encoder from Hugging Face.

Outputs (under `WORK / "runs"` — on Kaggle that is `/kaggle/working/runs/`):
- `run_dpr.json`
- `baseline_metrics.json`

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from setup.clerc_data import load_queries_and_qrels_hf
from setup.run_baselines import run_dpr
from setup.metrics import evaluate

# --- tuning ---
QUERY_TYPE = "direct"
QRELS_LEVEL = "passage"  # must match passage IDs from DPR
MAX_QUERIES = 200        # None = all test queries
DPR_CORPUS = 2_000      # passages from CLERC collection (increase for stronger baseline)
TOP_K = 100
OUT_DIR = WORK / "runs"  # from config cell; saves under /kaggle/working on Kaggle
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading queries and qrels...")
queries_df, qrels = load_queries_and_qrels_hf(QUERY_TYPE, QRELS_LEVEL)
if MAX_QUERIES:
    queries_df = queries_df.head(MAX_QUERIES)
    qids = set(queries_df["qid"])
    qrels = {q: qrels[q] for q in qids if q in qrels}

print("Running DPR (this may take a while)...")
run = run_dpr(queries_df, qrels, top_k=TOP_K, corpus_size=DPR_CORPUS)

ks = [1, 5, 10, 100]
metrics = evaluate(run, qrels, ks=ks)

with open(OUT_DIR / "run_dpr.json", "w") as f:
    json.dump(run, f, indent=2)
with open(OUT_DIR / "baseline_metrics.json", "w") as f:
    json.dump({"dpr": metrics, "config": {
        "query_type": QUERY_TYPE,
        "qrels_level": QRELS_LEVEL,
        "max_queries": MAX_QUERIES,
        "dpr_corpus": DPR_CORPUS,
        "top_k": TOP_K,
    }}, f, indent=2)

print("Metrics:", metrics)
print("Wrote:", OUT_DIR / "baseline_metrics.json")

## 7. (Optional) Run via CLI — same as local

Uncomment to run the full argument parser (writes under `--out-dir`).

In [ ]:
# Same as cell 6, but via CLI (uncomment). On Kaggle use: --out-dir /kaggle/working/runs
# !python -m setup.run_baselines --dpr --qrels-level passage --query-type direct --max-queries 200 --dpr-corpus 2000 --out-dir /kaggle/working/runs

## 8. Inspect results

In [ ]:
from pathlib import Path
import json

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path.cwd() / "kaggle_work")
p = WORK / "runs" / "baseline_metrics.json"
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))
else:
    print("No metrics file yet:", p)